# Statistical analysis of importance measures for Series-Parallel ensembles

This **single, self-contained** notebook consumes the two CSVs produced by the
experiments notebook and produces the statistical analyses.

* `importance_results_level1.csv` — one row per *(dataset, run, N)*; best SP tree on VAL.
* `importance_results_level2.csv` — one row per *(dataset, run, N)*; aggregated over **all** SP trees.

### Methodological principles
1. **Independent unit = dataset.** Every test runs on *dataset-level* aggregates,
   never on raw rows (10 runs x 6 sizes per dataset are not independent — testing on
   raw rows would be pseudoreplication and inflate significance).
2. **Tiny per-correlation samples** (each rho is over N = 3..8 models): we complement
   rho with a **top-1 hit-rate** and a **decision-regret** analysis that are
   well-behaved for small N, and we report **effect sizes and cluster-bootstrap CIs**
   rather than relying on the per-correlation p-values.
3. **Multiplicity** across measures is controlled with Holm.
4. **Level 2 is descriptive robustness only** (trees within a dataset are not
   independent), and is labelled as such.

We also separate two claims: **structural / cheap** measures (Birnbaum $I_B,I_{Bs},I_{Bf}$, Flip —
no test-time oracle; Flip needs no labels) vs **in-sample controls** (Oracle/Degradation on VAL,
which reuse the same operator as the target and mainly measure VAL->TEST stability).

## 0. Configuration

In [1]:
# ----------------------------- CONFIG -----------------------------
LEVEL1_CSV = "importance_results_level1.csv"
LEVEL2_CSV = "importance_results_level2.csv"

B_BOOT = 10000    # cluster-bootstrap resamples (CIs).  Lower to 2000 if slow.
ALPHA  = 0.05
SEED   = 0
MAKE_FIGURES = True
# ------------------------------------------------------------------

## 1. Analysis library
All statistics live in pure functions (numpy / pandas / scipy only).

In [2]:
"""
analysis_lib.py — Robust statistical analysis of importance-measure CSVs.

Operates on the two CSVs produced by the experiments notebook:
  - importance_results_level1.csv  (one row per dataset,run,N; best tree)
  - importance_results_level2.csv  (one row per dataset,run,N; over all trees)

Design principles (see paper plan):
  * Independent unit of analysis = DATASET (never raw rows).
  * Per-correlation sample is tiny (N=3..8 models) -> stratify by N,
    use permutation null, prefer top-1/regret over raw rho where possible.
  * Effect sizes + cluster-bootstrap CIs over p-values.
  * Multiplicity control (Holm) across measures.
  * Level 2 is descriptive robustness, NOT independent inference.
"""
import traceback
import json
import warnings
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, binomtest, friedmanchisquare, rankdata

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ config
MEASURES_L1 = ["birnbaum", "birn_success", "birn_failure", "flip", "oracle"] 
MEASURES_L2 = ["birnbaum", "birn_success", "birn_failure", "flip", "oracle"]

MEASURE_LABEL = {
    "birnbaum":     "Birnbaum total $I_B$",
    "birn_success": "Birnbaum success $I_{Bs}$",
    "birn_failure": "Birnbaum failure $I_{Bf}$",
    "flip":         "Flip",
    "oracle":       "Oracle (VAL)",
    "degrad":       "Degradation (VAL)",
}
# structural/cheap (do NOT use the test-time oracle operator) vs in-sample controls
CATEGORY = {
    "birnbaum": "structural", "birn_success": "structural",
    "birn_failure": "structural", "flip": "structural",
    "oracle": "control", "degrad": "control",
}
# per-row column prefixes for the per-model raw values in level1
PREFIX = {
    "birnbaum": "birn_", "birn_success": "birn_s_", "birn_failure": "birn_f_",
    "flip": "flip_", "oracle": "oracle_val_", "degrad": "degrad_",
}
GAIN_PREFIX = "gain_test_"


def _wilcoxon(x, alt="greater"):
    """Version-robust one-sample Wilcoxon (handles SciPy mode/method change)."""
    x = np.asarray(x, float); x = x[~np.isnan(x)]; x = x[x != 0]
    if len(x) < 1:
        return np.nan
    try:
        return wilcoxon(x, alternative=alt).pvalue
    except Exception:
        try:
            return wilcoxon(x, alternative=alt, mode="approx").pvalue
        except Exception:
            return np.nan

ALPHA = 0.05
# Nemenyi critical values q_alpha (Demsar 2006, Table 5), infinite df, two-tailed
NEMENYI_Q = {
    0.05: {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949,
           8: 3.031, 9: 3.102, 10: 3.164},
    0.10: {2: 1.645, 3: 2.052, 4: 2.291, 5: 2.459, 6: 2.589, 7: 2.693,
           8: 2.780, 9: 2.855, 10: 2.920},
}


# ------------------------------------------------------------------ helpers
def _round(x, n=5):
    if x is None:
        return None
    if isinstance(x, (float, np.floating)):
        return None if np.isnan(x) else round(float(x), n)
    return x


def detect_models(df1):
    """Return the global list of candidate model names from gain_test_* columns."""
    return sorted(c[len(GAIN_PREFIX):] for c in df1.columns
                  if c.startswith(GAIN_PREFIX))


def _row_models(row, all_models):
    """Models actually present (non-NaN gain) in a given level1 row."""
    return [m for m in all_models
            if (GAIN_PREFIX + m) in row.index and pd.notna(row[GAIN_PREFIX + m])]


def build_per_N(df1, measures=MEASURES_L1):
    """For each N, build aligned per-row arrays of importance, gains, dataset id.

    Returns dict: N -> {
        'imp':   {measure: ndarray (n_rows, N)},
        'gain':  ndarray (n_rows, N),
        'dataset': ndarray (n_rows,) of dataset names,
    }
    Rows keep their own model ordering; only within-row alignment matters.
    """
    all_models = detect_models(df1)
    out = {}
    for N, sub in df1.groupby("N"):
        sub = sub.reset_index(drop=True)
        n = len(sub)
        gain = np.full((n, N), np.nan)
        imp = {m: np.full((n, N), np.nan) for m in measures}
        ds = sub["dataset"].to_numpy()
        for r in range(n):
            row = sub.iloc[r]
            models = _row_models(row, all_models)
            if len(models) != N:
                # defensive: skip malformed rows
                continue
            for j, mod in enumerate(models):
                gain[r, j] = row[GAIN_PREFIX + mod]
                for m in measures:
                    col = PREFIX[m] + mod
                    if col in row.index:
                        imp[m][r, j] = row[col]
        out[N] = {"imp": imp, "gain": gain, "dataset": ds}
    return out


def _rank_rows(X):
    """Average-rank each row (ties handled), ignoring NaN-free assumption."""
    return np.apply_along_axis(rankdata, 1, X)


def spearman_rows(rank_imp, rank_gain):
    """Pearson-on-ranks per row == Spearman (ties via average ranks)."""
    a = rank_imp - rank_imp.mean(axis=1, keepdims=True)
    b = rank_gain - rank_gain.mean(axis=1, keepdims=True)
    num = (a * b).sum(axis=1)
    den = np.sqrt((a**2).sum(axis=1) * (b**2).sum(axis=1))
    return np.where(den > 1e-12, num / den, np.nan)


# ------------------------------------------------------------------ aggregation to dataset level
def dataset_level_rho(df1, measure, N=None, agg="median"):
    """One value per dataset: agg of per-row spearman_<measure> over rows.

    If N is given, restrict to that ensemble size.
    Returns Series indexed by dataset.
    """
    col = f"spearman_{measure}"
    sub = df1 if N is None else df1[df1["N"] == N]
    f = np.nanmedian if agg == "median" else np.nanmean
    g = sub.groupby("dataset")[col].apply(
        lambda s: f(s.values) if np.isfinite(s.values).any() else np.nan)
    return g.dropna()


# ------------------------------------------------------------------ one-sample inference
def cluster_bootstrap_ci(values, stat=np.median, B=10000, seed=0, alpha=0.05):
    """Percentile CI by resampling the (already dataset-level) values."""
    values = np.asarray(values, float)
    values = values[~np.isnan(values)]
    if len(values) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(B, len(values)))
    boot = stat(values[idx], axis=1)
    return (float(np.percentile(boot, 100 * alpha / 2)),
            float(np.percentile(boot, 100 * (1 - alpha / 2))))


def one_sample_vs_zero(values, alt="greater", B=10000, seed=0):
    """Test whether dataset-level values are centered above 0.

    Returns dict with n, median, mean, wilcoxon p, sign-test p,
    matched-pairs rank-biserial effect size, bootstrap CI of the median.
    """
    v = np.asarray(values, float)
    v = v[~np.isnan(v)]
    res = {"n_datasets": int(len(v)),
           "median": _round(np.median(v)) if len(v) else None,
           "mean": _round(np.mean(v)) if len(v) else None}
    if len(v) < 3:
        res.update({"wilcoxon_p": None, "sign_p": None,
                    "ci_low": None, "ci_high": None})
        return res
    nz = v[v != 0]
    w_p = _wilcoxon(v, alt)
    # sign test
    k = int((v > 0).sum())
    s_p = binomtest(k, len(nz) if len(nz) else len(v), 0.5,
                    alternative=alt).pvalue
    lo, hi = cluster_bootstrap_ci(v, np.median, B=B, seed=seed)
    res.update({"frac_pos": _round((v > 0).mean()),
                "wilcoxon_p": _round(w_p, 6), "sign_p": _round(s_p, 6),
                "ci_low": _round(lo), "ci_high": _round(hi)})
    return res


def holm(pdict, alpha=ALPHA):
    """Holm-Bonferroni on a {key: p} dict. Returns {key: (p_adj, reject)}."""
    items = [(k, v) for k, v in pdict.items() if v is not None and not np.isnan(v)]
    items.sort(key=lambda kv: kv[1])
    m = len(items)
    out, prev = {}, 0.0
    for i, (k, p) in enumerate(items):
        padj = min(1.0, max(prev, (m - i) * p))
        prev = padj
        out[k] = (_round(padj, 6), bool(padj < alpha))
    for k, v in pdict.items():
        if k not in out:
            out[k] = (None, False)
    return out


# ------------------------------------------------------------------ Friedman + Nemenyi across measures
def friedman_nemenyi(df1, measures=MEASURES_L1, agg="median"):
    """Block = dataset, treatment = measure. Demsar-style.

    Cell value = dataset-level aggregated rho (median over runs and N).
    """
    cols = {}
    for m in measures:
        s = dataset_level_rho(df1, m, N=None, agg=agg)
        cols[m] = s
    M = pd.DataFrame(cols).dropna()  # complete blocks only
    out = {"n_datasets_complete": int(len(M)), "measures": measures}
    if len(M) < 3 or M.shape[1] < 3:
        out["note"] = "insufficient complete blocks"
        return out, M
    stat, p = friedmanchisquare(*[M[m].values for m in measures])
    # mean ranks (higher rho -> rank 1)
    ranks = np.vstack([rankdata(-M.loc[i, measures].values) for i in M.index])
    mean_ranks = ranks.mean(axis=0)
    k, Nd = len(measures), len(M)
    out.update({"friedman_chi2": _round(stat), "friedman_p": _round(p, 8),
                "mean_ranks": {m: _round(r) for m, r in zip(measures, mean_ranks)}})
    for a in (0.05, 0.10):
        q = NEMENYI_Q[a].get(k)
        if q:
            cd = q * np.sqrt(k * (k + 1) / (6.0 * Nd))
            out[f"CD_{a}"] = _round(cd)
    # pairwise significance by |rank diff| > CD@0.05
    cd05 = out.get("CD_0.05")
    pairs = {}
    if cd05:
        for i in range(k):
            for j in range(i + 1, k):
                diff = abs(mean_ranks[i] - mean_ranks[j])
                pairs[f"{measures[i]}__vs__{measures[j]}"] = {
                    "rank_diff": _round(diff), "significant": bool(diff > cd05)}
    out["nemenyi_pairs"] = pairs
    return out, M


# ------------------------------------------------------------------ decision-utility / regret
def policy_analysis(per_N, measures=MEASURES_L1, B=10000, seed=0):
    """For each ensemble (row): which model to 'perfect'?

    Policies: argmax of each importance measure, plus random baseline.
    Metrics per row: realized TEST gain, regret = best-realized, top-1 hit.
    Aggregated to dataset level, then compared across datasets.
    """
    rng = np.random.default_rng(seed)
    rows = []  # collect per-row records
    for N, d in per_N.items():
        gain = d["gain"]
        ds = d["dataset"]
        valid = ~np.isnan(gain).any(axis=1)
        for r in np.where(valid)[0]:
            g = gain[r]
            best = np.nanmax(g)
            rec = {"dataset": ds[r], "N": int(N), "best": best,
                   "mean_gain": float(np.nanmean(g)),
                   "chance_top1": 1.0 / N}
            # random policy: expected realized gain & expected top1
            rec["realized_random"] = float(np.nanmean(g))
            n_best = int(np.sum(np.isclose(g, best)))
            rec["top1_random"] = n_best / N
            for m in measures:
                imp = d["imp"][m][r]
                if np.isnan(imp).any() or np.nanstd(imp) <= 1e-12:
                    # tie / undefined -> fall back to expected (random) choice
                    rec[f"realized_{m}"] = float(np.nanmean(g))
                    rec[f"top1_{m}"] = n_best / N
                    continue
                mx = np.nanmax(imp)
                tied = np.isclose(imp, mx)
                rec[f"realized_{m}"] = float(np.nanmean(g[tied]))
                rec[f"top1_{m}"] = float(np.mean(np.isclose(g[tied], best)))
            rows.append(rec)
    R = pd.DataFrame(rows)

    # aggregate to dataset level (mean over that dataset's ensembles)
    metrics = {}
    policies = ["random"] + measures
    ds_level = R.groupby("dataset").mean(numeric_only=True)
    for pol in policies:
        rg = ds_level[f"realized_{pol}"].values
        # regret relative to best, per dataset (best aggregated likewise)
        reg = (ds_level["best"] - ds_level[f"realized_{pol}"]).values
        t1 = ds_level[f"top1_{pol}"].values
        metrics[pol] = {
            "realized_gain_mean": _round(np.mean(rg)),
            "realized_gain_ci": [ _round(x) for x in
                                  cluster_bootstrap_ci(rg, np.mean, B=B, seed=seed)],
            "regret_mean": _round(np.mean(reg)),
            "regret_ci": [ _round(x) for x in
                           cluster_bootstrap_ci(reg, np.mean, B=B, seed=seed)],
            "top1_mean": _round(np.mean(t1)),
            "top1_ci": [ _round(x) for x in
                         cluster_bootstrap_ci(t1, np.mean, B=B, seed=seed)],
        }

    # paired comparisons: each measure-policy vs random (dataset-paired)
    comp = {}
    base_real = ds_level["realized_random"].values
    base_t1 = ds_level["top1_random"].values
    pvals_gain, pvals_t1 = {}, {}
    for m in measures:
        dr = ds_level[f"realized_{m}"].values - base_real
        dt = ds_level[f"top1_{m}"].values - base_t1
        pg = _wilcoxon(dr, "greater")
        pt = _wilcoxon(dt, "greater")
        comp[m] = {
            "d_gain_median": _round(np.median(dr)),
            "d_gain_ci": [ _round(x) for x in
                          cluster_bootstrap_ci(dr, np.median, B=B, seed=seed)],
            "gain_wilcoxon_p_vs_random": _round(pg, 6),
            "d_top1_median": _round(np.median(dt)),
            "top1_wilcoxon_p_vs_random": _round(pt, 6),
        }
        pvals_gain[m] = pg
        pvals_t1[m] = pt
    holm_gain = holm(pvals_gain)
    holm_t1 = holm(pvals_t1)
    for m in measures:
        comp[m]["gain_p_holm"] = holm_gain[m][0]
        comp[m]["gain_sig_holm"] = holm_gain[m][1]
        comp[m]["top1_p_holm"] = holm_t1[m][0]
        comp[m]["top1_sig_holm"] = holm_t1[m][1]

    # top-1 vs chance (per dataset chance = mean 1/N)
    chance = ds_level["chance_top1"].values
    top1_vs_chance = {}
    for pol in measures:
        d = ds_level[f"top1_{pol}"].values - chance
        p = _wilcoxon(d, "greater")
        top1_vs_chance[pol] = {"mean_chance": _round(np.mean(chance)),
                               "delta_median": _round(np.median(d)),
                               "wilcoxon_p": _round(p, 6)}
    return {"per_policy": metrics, "vs_random": comp,
            "top1_vs_chance": top1_vs_chance,
            "n_datasets": int(ds_level.shape[0])}


# ------------------------------------------------------------------ level 2 (descriptive)
def level2_descriptive(df2, measures=MEASURES_L2, B=10000, seed=0):
    out = {}
    for m in measures:
        med_col = f"spearman_{m}_median"
        fpos_col = f"spearman_{m}_frac_pos"
        ds_med = df2.groupby("dataset")[med_col].median().dropna()
        ds_fpos = df2.groupby("dataset")[fpos_col].mean().dropna()
        # exploratory dataset-level wilcoxon (labelled exploratory!)
        p = _wilcoxon(ds_med.values, "greater")
        out[m] = {
            "median_of_dataset_medians": _round(ds_med.median()),
            "median_rho_ci": [ _round(x) for x in
                               cluster_bootstrap_ci(ds_med.values, np.median, B=B, seed=seed)],
            "mean_frac_pos": _round(ds_fpos.mean()),
            "frac_pos_ci": [ _round(x) for x in
                             cluster_bootstrap_ci(ds_fpos.values, np.mean, B=B, seed=seed)],
            "exploratory_wilcoxon_p": _round(p, 6),
            "n_datasets": int(len(ds_med)),
        }
    # frac_pos by N (mean over datasets)
    byN = {}
    for N, sub in df2.groupby("N"):
        byN[int(N)] = {m: _round(sub[f"spearman_{m}_frac_pos"].mean())
                       for m in measures}
    out["_frac_pos_by_N"] = byN
    return out


C:\Users\jodivaso\AppData\Local\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 2. Load data and sanity-check the schema

In [3]:
import pandas as pd, numpy as np, json

df1 = pd.read_csv(LEVEL1_CSV)
try:
    df2 = pd.read_csv(LEVEL2_CSV)
except Exception as e:
    print("WARNING: level2 not loaded:", e); df2 = None

MEAS_L1 = [m for m in MEASURES_L1 if f"spearman_{m}" in df1.columns]
MEAS_L2 = ([m for m in MEASURES_L2 if f"spearman_{m}_median" in df2.columns]
           if df2 is not None else [])

per_N = build_per_N(df1, measures=MEAS_L1)

RESULTS = {"meta": {
    "n_datasets": int(df1["dataset"].nunique()),
    "n_runs": int(df1["run"].nunique()) if "run" in df1.columns
              else int(df1["random_state"].nunique()),
    "N_values": sorted(int(x) for x in df1["N"].unique()),
    "n_rows_level1": int(len(df1)),
    "n_rows_level2": (int(len(df2)) if df2 is not None else 0),
    "measures_level1": MEAS_L1, "measures_level2": MEAS_L2,
    "category": {m: CATEGORY[m] for m in MEAS_L1},
    "B_boot": B_BOOT, "alpha": ALPHA,
}}
print(json.dumps(RESULTS["meta"], indent=2))
print("\nLevel1 rows per N:", df1.groupby("N").size().to_dict())

{
  "n_datasets": 43,
  "n_runs": 10,
  "N_values": [
    3,
    4,
    5,
    6,
    7,
    8
  ],
  "n_rows_level1": 2562,
  "n_rows_level2": 2562,
  "measures_level1": [
    "birnbaum",
    "birn_success",
    "birn_failure",
    "flip",
    "oracle"
  ],
  "measures_level2": [
    "birnbaum",
    "birn_success",
    "birn_failure",
    "flip",
    "oracle"
  ],
  "category": {
    "birnbaum": "structural",
    "birn_success": "structural",
    "birn_failure": "structural",
    "flip": "structural",
    "oracle": "control"
  },
  "B_boot": 10000,
  "alpha": 0.05
}

Level1 rows per N: {3: 427, 4: 427, 5: 427, 6: 427, 7: 427, 8: 427}


## 3. Analysis A — Predictive validity (Level 1)
For each measure, one value per dataset = median over runs and N of the per-ensemble
Spearman rho between **importance on VAL** and **oracle gain on TEST**.

In [4]:
poolp = {}
RESULTS["A_predictivity_pooled"] = {}
for m in MEAS_L1:
    r = one_sample_vs_zero(dataset_level_rho(df1, m).values, B=B_BOOT, seed=SEED)
    RESULTS["A_predictivity_pooled"][m] = r
    poolp[m] = r["wilcoxon_p"]
hp = holm(poolp, alpha=ALPHA)
for m in MEAS_L1:
    RESULTS["A_predictivity_pooled"][m]["wilcoxon_p_holm"] = hp[m][0]
    RESULTS["A_predictivity_pooled"][m]["sig_holm"] = hp[m][1]

print(f"{'measure':14s} {'median_rho':>10s} {'95% CI':>20s} {'frac>0':>7s} {'p_Holm':>9s}  cat")
for m in MEAS_L1:
    d = RESULTS["A_predictivity_pooled"][m]
    print(f"{m:14s} {d['median']:>10.3f}  [{d['ci_low']:.3f},{d['ci_high']:.3f}]"
          f"   {d['frac_pos']:>5.0%}  {str(d['wilcoxon_p_holm']):>9s}  {CATEGORY[m]}")

measure        median_rho               95% CI  frac>0    p_Holm  cat
birnbaum            0.713  [0.647,0.772]     93%      4e-06  structural
birn_success        0.800  [0.778,0.839]     98%        0.0  structural
birn_failure        0.801  [0.734,0.862]     95%      4e-06  structural
flip                0.953  [0.923,0.981]     98%        0.0  structural
oracle              0.963  [0.935,0.990]    100%        0.0  control


## 4. Analysis C — Friedman + Nemenyi across measures (Demsar style)
Block = dataset, treatment = measure (dataset-level aggregated rho). Friedman omnibus,
mean ranks, and Nemenyi.

In [5]:
fr, _M = friedman_nemenyi(df1, measures=MEAS_L1)
RESULTS["C_friedman_nemenyi"] = fr
print("Friedman chi2 =", fr.get("friedman_chi2"), " p =", fr.get("friedman_p"),
      f"({fr.get('n_datasets_complete')} complete blocks)")
print("CD@0.05 =", fr.get("CD_0.05"))
print("Mean ranks (1 = best):")
for m, r in sorted(fr.get("mean_ranks", {}).items(), key=lambda kv: kv[1]):
    print(f"   {m:14s} {r:.3f}")

Friedman chi2 = 112.9219  p = 0.0 (40 complete blocks)
CD@0.05 = 0.96449
Mean ranks (1 = best):
   oracle         1.525
   flip           1.875
   birn_success   3.225
   birn_failure   3.650
   birnbaum       4.725


## 5. Analysis D — Decision-utility / regret (the practical headline)
Each ensemble is a decision: *which base model should I invest in perfecting?*
Each measure picks `argmax`; baseline is a **random** model. We report realized TEST
oracle gain, **regret** = (best possible - realized), and **top-1 hit-rate** (did the
measure point at the truly most-valuable model?), aggregated to dataset level with
cluster-bootstrap CIs, plus dataset-paired Wilcoxon vs random (Holm) and top-1 vs chance.

In [6]:
RESULTS["D_policy"] = policy_analysis(per_N, measures=MEAS_L1, B=B_BOOT, seed=SEED)
pp = RESULTS["D_policy"]["per_policy"]; vs = RESULTS["D_policy"]["vs_random"]
print(f"{'policy':14s} {'realized':>9s} {'regret':>8s} {'top1':>7s} "
      f"{'dgain p_Holm':>13s} {'top1 sig':>9s}")
print(f"{'random':14s} {pp['random']['realized_gain_mean']:>9.4f} "
      f"{pp['random']['regret_mean']:>8.4f} {pp['random']['top1_mean']:>7.2%}")
for m in MEAS_L1:
    print(f"{m:14s} {pp[m]['realized_gain_mean']:>9.4f} {pp[m]['regret_mean']:>8.4f} "
          f"{pp[m]['top1_mean']:>7.2%} {str(vs[m]['gain_p_holm']):>13s} "
          f"{str(vs[m]['top1_sig_holm']):>9s}")

policy          realized   regret    top1  dgain p_Holm  top1 sig
random            0.0702   0.0698  29.38%
birnbaum          0.1228   0.0173  63.17%           0.0      True
birn_success      0.1279   0.0121  68.02%           0.0      True
birn_failure      0.1226   0.0175  63.58%           0.0      True
flip              0.1388   0.0013  88.40%           0.0      True
oracle            0.1389   0.0011  87.67%           0.0      True


## 6. Analysis E — Level 2 robustness across all SP trees (descriptive)
Distribution over datasets of the per-(dataset,N) median rho and of the fraction of trees
with rho > 0, aggregated to dataset level.

In [7]:
if df2 is not None and MEAS_L2:
    RESULTS["E_level2_descriptive"] = level2_descriptive(df2, measures=MEAS_L2,
                                                         B=B_BOOT, seed=SEED)
    print(f"{'measure':14s} {'median_rho':>10s} {'95% CI':>20s} {'mean frac>0':>12s}")
    for m in MEAS_L2:
        d = RESULTS["E_level2_descriptive"][m]
        print(f"{m:14s} {d['median_of_dataset_medians']:>10.3f}  "
              f"[{d['median_rho_ci'][0]:.3f},{d['median_rho_ci'][1]:.3f}]   "
              f"{d['mean_frac_pos']:>11.2%}")
else:
    RESULTS["E_level2_descriptive"] = None
    print("Level 2 skipped (no CSV).")

measure        median_rho               95% CI  mean frac>0
birnbaum            0.714  [0.643,0.771]        89.17%
birn_success        0.800  [0.786,0.829]        94.81%
birn_failure        0.714  [0.607,0.786]        88.70%
flip                0.902  [0.900,0.954]        97.12%
oracle              0.964  [0.943,0.976]        98.85%


## 7. Some figures (saved as PDF)

In [8]:
if MAKE_FIGURES:
    try:
        import matplotlib; matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        LAB = {"birnbaum":"$I_B$","birn_success":"$I_{Bs}$","birn_failure":"$I_{Bf}$",
               "flip":"Flip","oracle":"Oracle","degrad":"Degrad"}

        fig, ax = plt.subplots(figsize=(6,3.2))
        for y, m in enumerate(MEAS_L1):
            d = RESULTS["A_predictivity_pooled"][m]
            ax.errorbar(d["median"], y, fmt="o", capsize=3,
                        xerr=[[d["median"]-d["ci_low"]],[d["ci_high"]-d["median"]]])
        ax.axvline(0, ls=":", c="gray"); ax.set_yticks(range(len(MEAS_L1)))
        ax.set_yticklabels([LAB.get(m,m) for m in MEAS_L1])
        ax.set_xlabel("Dataset-level median Spearman rho (VAL imp -> TEST gain)")
        ax.set_title("Predictive validity (95% cluster-bootstrap CI)")
        plt.tight_layout(); plt.savefig("fig_predictivity.pdf"); plt.close()

        pp = RESULTS["D_policy"]["per_policy"]; order = ["random"]+MEAS_L1
        fig, ax = plt.subplots(figsize=(6,3.2))
        ax.bar(range(len(order)), [pp[p]["regret_mean"] for p in order],
               yerr=[[pp[p]["regret_mean"]-pp[p]["regret_ci"][0] for p in order],
                     [pp[p]["regret_ci"][1]-pp[p]["regret_mean"] for p in order]], capsize=3)
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(["Random"]+[LAB.get(m,m) for m in MEAS_L1], rotation=30)
        ax.set_ylabel("Mean regret (TEST F1)")
        ax.set_title("Decision-utility: regret of importance-guided model selection")
        plt.tight_layout(); plt.savefig("fig_regret.pdf"); plt.close()

        if RESULTS.get("E_level2_descriptive"):            
            byN = RESULTS["E_level2_descriptive"]["_frac_pos_by_N"]
#             Ns = sorted(int(k) for k in byN)
#             fig, ax = plt.subplots(figsize=(6,3.2))
#             for m in MEAS_L2:
#                 ax.plot(Ns, [byN[str(n)][m] for n in Ns], marker="o", label=LAB.get(m,m))
            Ns = sorted(byN.keys())  # Mantén como strings
            fig, ax = plt.subplots(figsize=(6,3.2))
            for m in MEAS_L2:
                ax.plot(Ns, [byN[n][m] for n in Ns], marker="o", label=LAB.get(m,m))
            ax.axhline(0.5, ls=":", c="gray"); ax.set_xlabel("N (ensemble size)")
            ax.set_ylabel("Frac. trees with rho>0"); ax.legend(fontsize=7)
            ax.set_title("Level 2 robustness across all SP trees")
            plt.tight_layout(); plt.savefig("fig_level2.pdf"); plt.close()
            print("Fig_level2 saved")
        print("Figures saved: fig_predictivity.pdf, fig_regret.pdf, fig_level2.pdf")
    except Exception as e:
        print("Figure generation skipped:", e)
        traceback.print_exc()

Fig_level2 saved
Figures saved: fig_predictivity.pdf, fig_regret.pdf, fig_level2.pdf
